In [ ]:
# ==============================
# GOLD LAYER - BUSINESS ANALYTICS (DATE PARTITION)
# ==============================

from pyspark.sql.functions import col, sum, month, year, when, count

# ==============================
# STEP 1: READ FROM SILVER
# ==============================

sales = spark.read.table("silver_sales")
exchange = spark.read.table("silver_exchange_rate")

# ==============================
# STEP 2: PREPARE DATA
# ==============================

sales = sales.withColumn("month", month(col("order_date"))) \
             .withColumn("year", year(col("order_date")))

# ==============================
# STEP 3: JOIN DATA
# ==============================

df_join = sales.join(
    exchange,
    on=["month", "year"],
    how="left"
)

# ==============================
# STEP 4: KPI 1 - REVENUE VND
# ==============================

df_revenue = df_join.withColumn(
    "revenue_vnd",
    col("total_amount") * col("exchange_rate")
)

revenue_monthly = df_revenue.groupBy("year", "month", "items") \
    .agg(sum("revenue_vnd").alias("total_revenue_vnd"))

# ==============================
# STEP 5: KPI 2 - PROMOTION EFFECTIVENESS
# ==============================

promo_analysis = sales.withColumn(
    "is_discount",
    when(col("discount_code").isNotNull(), 1).otherwise(0)
)

promo_result = promo_analysis.groupBy("location") \
    .agg(
        (sum("is_discount") / count("*") * 100).alias("discount_percentage")
    )

# ==============================
# STEP 6: SAVE GOLD (PARTITION BY DATE)
# ==============================

# 👉 Thêm cột ingestion_date
revenue_monthly = revenue_monthly.withColumn("ingestion_date", col("year"))
promo_result = promo_result.withColumn("ingestion_date", col("location"))

# 👉 Ghi ra Gold layer theo ngày (có thể đổi lại hardcode nếu muốn)
revenue_monthly.write.mode("overwrite").saveAsTable("gold_revenue_monthly")

promo_result.write.mode("overwrite").saveAsTable("gold_promotion_effectiveness")

# ==============================
# STEP 7: VALIDATION
# ==============================

print("Revenue rows:", revenue_monthly.count())
print("Promo rows:", promo_result.count())

# ==============================
# EXIT PIPELINE
# ==============================

mssparkutils.notebook.exit("Success")

StatementMeta(, , -1, SessionError, , SessionError, True)

InvalidHttpRequest: [TooManyRequestsForCapacity] [TooManyRequestsForCapacity] HTTP Response code 430: This Spark job can’t be run because you’ve hit a Spark compute or API rate limit. To proceed, cancel an active Spark job through the Monitoring hub, choose a larger capacity SKU, or try again later. For more visibility and control, go to Workspace settings → Job management (Job Concurrency & Queue Monitoring) to review running and queued Spark jobs, understand capacity contention, and take action as needed. [Learn more at 'https://go.microsoft.com/fwlink/?linkid=2356970&clcid=0x409']. HTTP status code: 430.

In [ ]:
spark.read.table("gold_revenue_monthly").show()